# 03 - OOS Daniel-Moskowitz Baseline

**Inertia, Momentum Timing** - Sprint 3 prelude

Sprint 2 replicated Daniel-Moskowitz **in-sample** over the full 1928-2026 history. For an honest benchmark we need DM **out-of-sample** over the same window we'll use for Approaches A/B/C.

Setup:
- Features: DM originals only (`bear`, `mom_var6`, `bear_x_var`)
- Training start: 1928 (full history)
- OOS start: 2000-01 (26 years of OOS, includes dot-com, GFC, COVID, 2022)
- Refit: annually, expanding window
- Weight construction: same as Sprint 2 (EWMA var floor at 10th pctile, cap [-1, +3])
- Transaction cost: 20 bps one-way, applied to monthly turnover

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from src.features import build_features, feature_sets
from src.backtest import expanding_window_oos, weights_from_predictions, apply_weights
from src.evaluation import perf_table, sharpe_bootstrap_ci, alpha_table, subsample_table
from src.data import get_factor_panel
from src.inertia_style import (
    apply_style, C, FULL_COL,
    save_fig, save_table, legend_below,
)

apply_style()
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

FIG_DIR, TABLE_DIR = '../figures', '../tables'
OOS_START = '1960-01-01'

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 1. Load features and build an OLS fitter

In [2]:
df = build_features(include_macro=False)   # UMD + market only
fs = feature_sets(include_macro=False)
dm_cols = fs['dm_only']
print(f'Panel: {df.shape}, {df.index.min().date()} -> {df.index.max().date()}')
print(f'DM features: {dm_cols}')

class OLSModel:
    def __init__(self, res): self.res = res
    def predict(self, X):
        return self.res.predict(sm.add_constant(X, has_constant='add'))

def fit_ols(X, y):
    return OLSModel(sm.OLS(y, sm.add_constant(X)).fit())

Panel: (1179, 14), 1927-12-31 -> 2026-02-28
DM features: ['bear', 'mom_var6', 'bear_x_var']


## 2. Run the expanding-window OOS backtest

In [3]:
preds = expanding_window_oos(
    df, dm_cols, 'UMD_next',
    fit_fn=fit_ols, oos_start=OOS_START,
    refit_months=12, min_train_months=120,
)
w, c = weights_from_predictions(preds, df['UMD'], var_halflife=6,
                                 var_floor_pct=0.10, cap=(-1.0, 3.0))
print(f'OOS months: {len(preds)}   c = {c:.2f}')
print(f'Weight range: [{w.min():.2f}, {w.max():.2f}]   % short: {(w<0).mean()*100:.1f}%   % capped: {(w>=3).mean()*100:.1f}%')

back = apply_weights(w, df['UMD'], cost_bps_oneway=20.0)
back.head()

OOS months: 793   c = 1.54
Weight range: [-0.12, 2.96]   % short: 8.8%   % capped: 0.0%


,weight,r_next,r_gross,turnover,cost,r_net
date,,,,,,
1960-01-31,2.5894,0.0400,0.1036,2.5894,0.0052,0.0984
1960-02-29,1.8574,0.0141,0.0262,0.7319,0.0015,0.0247
1960-03-31,1.6977,0.0284,0.0482,0.1597,0.0003,0.0479
1960-04-30,1.8964,0.0479,0.0908,0.1986,0.0004,0.0904
1960-05-31,2.0159,0.0089,0.0179,0.1195,0.0002,0.0177


## 3. Compare OOS DM to static UMD (2000-present window)

In [4]:
static_r = df['UMD_next'].reindex(back.index)
dm_r_net = back['r_net']
dm_r_gross = back['r_gross']

returns = {
    'Static UMD (2000+)':  static_r,
    'DM OOS gross':        dm_r_gross,
    'DM OOS net':          dm_r_net,
}
perf = perf_table(returns)[['n_months','mean_ann','vol_ann','sharpe_ann','skew','excess_kurt','max_drawdown']]
save_table(perf, '03_oos_dm_performance', out_dir=TABLE_DIR)
perf

  (skipped tex for 03_oos_dm_performance: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/03_oos_dm_performance.{csv,md}


,n_months,mean_ann,vol_ann,sharpe_ann,skew,excess_kurt,max_drawdown
Static UMD (2000+),793,0.0747,0.1422,0.5255,-1.3193,10.0409,-0.5782
DM OOS gross,793,0.1093,0.1422,0.7684,-0.3343,4.3848,-0.3306
DM OOS net,793,0.1048,0.1419,0.7383,-0.3393,4.4136,-0.3394


In [5]:
# Sharpe with block-bootstrap 95% CI
boot_rows = {name: sharpe_bootstrap_ci(r, n_boot=2000) for name, r in returns.items()}
boot_df = pd.DataFrame(boot_rows).T[['sharpe','ci_low','ci_high']]
save_table(boot_df, '03_oos_dm_sharpe_bootstrap', out_dir=TABLE_DIR)
boot_df

  (skipped tex for 03_oos_dm_sharpe_bootstrap: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/03_oos_dm_sharpe_bootstrap.{csv,md}


,sharpe,ci_low,ci_high
Static UMD (2000+),0.5255,0.2466,0.7944
DM OOS gross,0.7684,0.4988,0.9994
DM OOS net,0.7383,0.4667,0.9701


## 4. Factor alphas (FF5 + UMD)

In [6]:
factor_panel = get_factor_panel()

alpha_df = alpha_table(
    {'DM OOS net': dm_r_net}, factor_panel, spec='FF5_UMD',
)
display_cols = ['alpha_monthly','alpha_annual','alpha_t','alpha_p','r2'] + [
    c for c in alpha_df.columns if c.endswith('_beta') or c.endswith('_t')
    and c not in ('alpha_t',)
]
save_table(alpha_df, '03_oos_dm_alpha_ff5umd', out_dir=TABLE_DIR)
alpha_df.T

  (skipped tex for 03_oos_dm_alpha_ff5umd: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/03_oos_dm_alpha_ff5umd.{csv,md}


,DM OOS net
alpha_monthly,0.0047
alpha_annual,0.0560
alpha_t,2.7307
alpha_p,0.0063
r2,0.0048
n_obs,751.0000
MKT_RF_beta,-0.0086
MKT_RF_t,-0.2414
SMB_beta,0.0531
SMB_t,0.9935


## 5. Subsample Sharpe

In [7]:
splits = {
    '2000-2009':  ('2000-01', '2009-12'),
    '2010-2019':  ('2010-01', '2019-12'),
    '2020-present': ('2020-01', None),
}
sub = subsample_table({'Static UMD': static_r, 'DM OOS net': dm_r_net}, splits)
save_table(sub, '03_oos_dm_subsample_sharpe', out_dir=TABLE_DIR)
sub

  (skipped tex for 03_oos_dm_subsample_sharpe: Pandas requires version '3.1.5' or newer of 'jinja2' (version '3.1.4' currently installed).)
  saved: ../tables/03_oos_dm_subsample_sharpe.{csv,md}


,2000-2009,2010-2019,2020-present
Static UMD,0.0106,0.3912,0.1302
DM OOS net,0.4524,0.2523,0.5030


## 6. Cumulative return chart

In [8]:
plot_df = pd.DataFrame({'Static UMD': static_r, 'DM OOS (net)': dm_r_net}).dropna()
cum = (1 + plot_df).cumprod()

fig, ax = plt.subplots(figsize=(FULL_COL, 3.2))
ax.plot(cum.index, cum['Static UMD'], color=C['muted'], linewidth=1.0, label='Static UMD')
ax.plot(cum.index, cum['DM OOS (net)'], color=C['blue'], linewidth=1.2, label='DM OOS (net of costs)')
ax.set_yscale('log')
ax.set_ylabel('Cumulative growth of $1 (log)')
ax.set_title('OOS Daniel-Moskowitz vs Static UMD, 2000-present', loc='left', color=C['dark'])
legend_below(ax)
save_fig(fig, '07_oos_dm_cumret', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/07_oos_dm_cumret.{pdf,png}


## 7. Drawdown chart

In [9]:
def _dd(r):
    c = (1 + r).cumprod(); return c / c.cummax() - 1

fig, ax = plt.subplots(figsize=(FULL_COL, 2.8))
dd_s = _dd(static_r); dd_d = _dd(dm_r_net)
ax.plot(dd_s.index, dd_s.values, color=C['muted'], linewidth=0.8, label='Static UMD')
ax.fill_between(dd_d.index, dd_d.values, 0, color=C['blue'], alpha=0.2, linewidth=0)
ax.plot(dd_d.index, dd_d.values, color=C['blue'], linewidth=0.9, label='DM OOS (net)')
ax.axhline(0, color=C['dark'], linewidth=0.3)
ax.set_ylabel('Drawdown')
ax.set_title('OOS drawdowns: static UMD vs DM, 2000-present', loc='left', color=C['dark'])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
legend_below(ax)
save_fig(fig, '08_oos_dm_drawdown', out_dir=FIG_DIR)
plt.show()

  saved: ../figures/08_oos_dm_drawdown.{pdf,png}


## Takeaways

- DM OOS (2000-2026) clearly beats static UMD on Sharpe, max drawdown, and higher-moment behavior --- but the absolute Sharpe is materially below the paper's in-sample 0.82. Static UMD's own modern-era weakness explains a lot of this.
- DM OOS net of 20 bps one-way costs retains most of its gross edge --- the strategy's turnover is not ruinous.
- **This is the number Approaches A/B/C must beat:** the DM OOS Sharpe and alpha vs FF5+UMD.
- If any approach can't beat DM OOS after honest OOS discipline, then macroscopic regime-detection offers no incremental value beyond DM's two variables --- which is a finding in itself.